# MACH MCP Server — Full HTTP Example

Complete walkthrough of all MACH MCP tools and resources using the **SSE/HTTP transport**.  
Requires the MACH backend to be running (locally or deployed).

```bash
# Start the server locally
uv run uvicorn app.main:app --reload
```

The MCP server is mounted at `/mcp` — so the SSE endpoint is `http://localhost:8000/mcp/sse`.

In [ ]:
%pip install -q fastmcp

In [ ]:
import asyncio, json
from fastmcp import Client

MCP_URL = "http://localhost:8000/mcp/sse"  # swap for deployed URL when live

async def call(tool: str, **kwargs):
    async with Client(MCP_URL) as c:
        result = await c.call_tool(tool, kwargs)
        return result.data

async def read(uri: str):
    async with Client(MCP_URL) as c:
        result = await c.read_resource(uri)
        return json.loads(result[0].text)

## 1. Discover what tools and resources are available

In [ ]:
async def list_all():
    async with Client(MCP_URL) as c:
        tools     = await c.list_tools()
        resources = await c.list_resources()
    print("Tools:",     [t.name for t in tools])
    print("Resources:", [str(r.uri) for r in resources])

asyncio.run(list_all())

## 2. `search_catalog` — find entries by keyword

In [ ]:
results = asyncio.run(call("search_catalog", query="segmentation"))
for r in results:
    print(f"[{r['judgement']}] {r['name']} ({r['category']}) — {r['judgement_reason']}")

In [ ]:
# Narrow by category
results = asyncio.run(call("search_catalog", query="FHIR", category="specs", limit=5))
for r in results:
    print(f"{r['name']} — {r['governance']} governance, maturity: {r['maturity']}")

## 3. `get_entry` — full details for a specific entry

In [ ]:
entry = asyncio.run(call("get_entry", category="ai-ml-models", identifier="hover-net"))

print(f"Name        : {entry['name']}")
print(f"Judgement   : {entry['judgement']} — {entry['judgement_reason']}")
print(f"License     : {entry['license']}")
print(f"URL         : {entry['url']}")
print(f"Domains     : {entry['clinical_domain']}")
print(f"Deployment  : {entry['deployment_context']}")
print()
print("Evidence:")
for e in entry['evidence']:
    print(f"  [{e['evidence_type']}] {e['label']} — {e['url']}")

## 4. `list_entries` — browse with filters

In [ ]:
# All Adopt entries
adopt = asyncio.run(call("list_entries", judgement="Adopt"))
print(f"{adopt['total']} Adopt entries:")
for e in adopt['entries']:
    print(f"  {e['name']} ({e['category']})")

In [ ]:
# Combine filters: datasets with foundation governance
filtered = asyncio.run(call("list_entries", category="datasets", governance="academic"))
print(f"{filtered['total']} academic datasets:")
for e in filtered['entries']:
    print(f"  {e['name']} — {e['maturity']}")

## 5. Resources — catalog stats and categories

In [ ]:
stats = asyncio.run(read("mach://catalog/stats"))
print(f"Total entries: {stats['total_entries']}")
print()
for cat, counts in stats['by_category'].items():
    adopt   = counts.get('Adopt', 0)
    assess  = counts.get('Assess', 0)
    caution = counts.get('Caution', 0)
    print(f"  {cat:<18} total={counts['total']}  Adopt={adopt}  Assess={assess}  Caution={caution}")

In [ ]:
cats = asyncio.run(read("mach://catalog/categories"))
for name, desc in cats['categories'].items():
    print(f"{name:<18} {desc}")